# PSELDNets × v9（最終土俵: 6クラス・1188本・絶対較正・歩行マイク混在）

事前準備（ローカル→Drive）: `dataset_outdoor_siren_v9.zip`（約1.4GB）を
`MyDrive/PSELDNets_data/` に置く。

- クラス6 = Siren / Horn / BackupBeep / BikeBell / CarDrive / **Crossing(踏切)**
- 学習 fold1_room1 640本 / val fold2_room1 240本 / test fold3_room1 240本（**testは最終1回まで触らない**）
- 追加枠: 交差点サイレン fold2_room9 20本・レベル正規化プローブ fold9_room1 48本（主表とは別枠で推論だけ行う）
- 設計の正: `out/v9_design_v2_2026-07-16.md`。**データを変えたら EXP_NAME を必ず変える**

---
## 1. GPU 確認

In [ ]:
import torch
assert torch.cuda.is_available(), '⚠ GPUがありません。ランタイム→T4 GPU を選択してください'
print(f'PyTorch : {torch.__version__}')
print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Drive マウントと設定

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ==== 設定 ====
DRIVE_DATA = '/content/drive/MyDrive/PSELDNets_data'
DRIVE_LOGS = '/content/drive/MyDrive/PSELDNets_logs'
DRIVE_CKPT = '/content/drive/MyDrive/PSELDNets_ckpts'
DATASET    = 'outdoor_siren_v9'
EXP_NAME   = 'outdoor_siren_v9_run1'

import os
for d in [DRIVE_DATA, DRIVE_LOGS, DRIVE_CKPT]:
    os.makedirs(d, exist_ok=True)
V9_ZIP = f'{DRIVE_DATA}/dataset_outdoor_siren_v9.zip'
assert os.path.exists(V9_ZIP), '⚠ dataset_outdoor_siren_v9.zip がDriveにありません'
print(f'OK: v9 zip {os.path.getsize(V9_ZIP)/1e6:.0f}MB')

## 3. リポジトリ clone

In [ ]:
import os

REPO = '/content/PSELDNets'

if not os.path.exists(f'{REPO}/src'):
    !git clone https://github.com/Jinbo-Hu/PSELDNets {REPO}
else:
    print(f'既に存在: {REPO}')

os.chdir(REPO)
print(f'CWD: {os.getcwd()}')

## 4. 依存インストール

In [ ]:
!pip install -q \
    librosa \
    soundfile \
    lightning==2.2.1 \
    hydra-core==1.3.2 \
    hydra-colorlog==1.2.0 \
    hydra-joblib-launcher==1.2.0 \
    torchmetrics==1.3.1

import numpy, lightning, torchmetrics, librosa
print(f'numpy {numpy.__version__} / lightning {lightning.__version__} / '
      f'torchmetrics {torchmetrics.__version__} / librosa {librosa.__version__}')

## 5. 事前学習チェックポイント（Drive キャッシュ → なければ HF から）

In [ ]:
import shutil

os.makedirs('ckpts', exist_ok=True)
CKPT = 'ckpts/mACCDOA-HTSAT-0.567.ckpt'
CACHE = f'{DRIVE_CKPT}/mACCDOA-HTSAT-0.567.ckpt'

if not os.path.exists(CKPT):
    if os.path.exists(CACHE):
        print('Drive キャッシュからコピー...')
        shutil.copy(CACHE, CKPT)
    else:
        print('HuggingFace からダウンロード...')
        from huggingface_hub import hf_hub_download
        src = hf_hub_download(repo_id='Jinbo-HU/PSELDNets',
                              filename='model/mACCDOA-HTSAT-0.567.ckpt',
                              repo_type='dataset')
        shutil.copy(src, CKPT)
        shutil.copy(CKPT, CACHE)   # 次回用に Drive にキャッシュ
print(f'OK: {CKPT} ({os.path.getsize(CKPT)/1e6:.0f} MB)')

## 6. データセット展開（1.4GB、数分かかる）

In [ ]:
import zipfile, os

if not os.path.exists(f'datasets/{DATASET}/foa'):
    with zipfile.ZipFile(V9_ZIP) as z:
        z.extractall('.')
    print('v9 unzipped')
else:
    print('展開済み')

n_foa = len(os.listdir(f'datasets/{DATASET}/foa'))
n_meta = len(os.listdir(f'datasets/{DATASET}/metadata'))
n_cls = len(open('datasets/cls_indices_train.tsv').readlines())
print(f'foa: {n_foa} / metadata: {n_meta} / classes: {n_cls}')
assert n_foa in (1188, 1187) and n_meta == n_foa and n_cls == 6, '⚠ 中身が想定と違います'  # 1187=無音1本除外後

### 6b. 無音クリップ1本の除外（v9パッチ・必ず前処理の前に実行）

fold1_room1_mix035 は「車が最後まで暗騒音に埋もれ、可聴フレームがゼロ」の物理的に正しい
負例で、ラベルCSVが空になる。PSELDNetsのラベル抽出は空CSVを読めず前処理が途中で落ちる
（→ adpitラベルh5が作られず学習が `FileNotFoundError` になる）。該当は学習用fold1の
1本だけなので学習セットから除外する（val/testは無傷。ローカルの解析では1188本全部を使う）。

In [ ]:
# 空ラベルの無音クリップ(学習fold1の1本)を除外し、壊れた前処理の残骸を消す
import os, shutil
for f in [f'datasets/{DATASET}/foa/fold1_room1_mix035.flac',
          f'datasets/{DATASET}/metadata/fold1_room1_mix035.csv']:
    if os.path.exists(f):
        os.remove(f)
        print('removed', f)
shutil.rmtree('_hdf5', ignore_errors=True)
print('cleaned _hdf5 -> 次にセル8(前処理)を実行してください')

## 7. 設定ファイル（本体＋推論3種: val / 交差点 / プローブ）

rooms フィルタは部分文字列マッチ: `[fold2_room1]` は room9 を含まない。
追加枠（room9・fold9）は学習にも val にも入らず、専用の推論だけで使う。

In [ ]:
data_yaml = """audio_type: foa
audio_feature: logmelIV
sample_rate: 24000
nfft: 1024
n_mels: 64
hoplen: 240
window: hann

train_chunklen_sec: 10
train_hoplen_sec: 10
test_chunklen_sec: 10
test_hoplen_sec: 10

train_dataset:
  outdoor_siren_v9: [fold1_room1]
valid_dataset:
  outdoor_siren_v9: [fold2_room1]
test_dataset:
  outdoor_siren_v9: [fold3_room1]
"""

def variant(rooms):
    return data_yaml.replace(
        'test_dataset:\n  outdoor_siren_v9: [fold3_room1]',
        f'test_dataset:\n  outdoor_siren_v9: [{rooms}]')

exp_yaml = """# @package _global_
defaults:
 - override /data: outdoor_siren_v9.yaml
 - override /loss: multi_accdoa.yaml
 - _self_

task_name: outdoor_siren_v9

model:
  batch_size: 8
  kwargs:
    pretrained_path: ckpts/mACCDOA-HTSAT-0.567.ckpt
    audioset_pretrain: false
  optimizer:
    kwargs: {lr: 0.0003}
  lr_scheduler:
    kwargs: {step_size: 60}

trainer:
  max_epochs: 100
  check_val_every_n_epoch: 5
"""

open('configs/data/outdoor_siren_v9.yaml', 'w').write(data_yaml)
open('configs/experiment/outdoor_siren_v9.yaml', 'w').write(exp_yaml)
for tag, rooms in [('valinfer', 'fold2_room1'),
                   ('scenario', 'fold2_room9'),
                   ('probe', 'fold9_room1')]:
    open(f'configs/data/outdoor_siren_v9_{tag}.yaml', 'w').write(variant(rooms))
    open(f'configs/experiment/outdoor_siren_v9_{tag}.yaml', 'w').write(
        exp_yaml.replace('override /data: outdoor_siren_v9.yaml',
                         f'override /data: outdoor_siren_v9_{tag}.yaml'))
print('wrote configs (v9 + valinfer/scenario/probe)')

## 8. 前処理 → HDF5（初回のみ、10分前後）

In [ ]:
IDX = f'_hdf5/data/24000fs/wav/dev/{DATASET}_10sChunklen_10sHoplen_train.csv'
if not os.path.exists(IDX):
    !python src/preproc.py dataset={DATASET}
else:
    print('前処理済み')
!head -3 {IDX}

## 9. 最終チェック

In [ ]:
checks = [
    ('ckpts/mACCDOA-HTSAT-0.567.ckpt',     '事前学習チェックポイント'),
    ('datasets/cls_indices_train.tsv',      'クラス辞書 TSV (6クラス)'),
    (f'datasets/{DATASET}/foa',             'FOA 音声 (1188)'),
    (f'datasets/{DATASET}/metadata',        'ラベル CSV (1188)'),
    (f'configs/experiment/{DATASET}.yaml',  '実験設定'),
    (IDX,                                   '前処理インデックス'),
]
for path, name in checks:
    ok = os.path.exists(path) and (not os.path.isdir(path) or len(os.listdir(path)) > 0)
    print(f'  [{"OK" if ok else "NG"}] {name}')

## 10. 学習（T4 で 1〜2 時間 / 100epoch）

- 途中で切れても再実行すれば `last.ckpt` から自動再開（Drive 永続化）
- val は fold2_room1 のみ。test(fold3) はここでは一切使わない
- **データを変えて学習し直すときは必ず EXP_NAME を変えること**

In [ ]:
LAST = f'{DRIVE_LOGS}/{DATASET}/runs/{EXP_NAME}/checkpoints/last.ckpt'
resume = f'ckpt_path={LAST}' if os.path.exists(LAST) else ''
print('resume:', resume or '(new run)')

!python src/train.py experiment={DATASET} \
    experiment_name={EXP_NAME} \
    paths.log_dir={DRIVE_LOGS} \
    {resume}

## 11. 学習曲線（val 抜粋）

In [ ]:
import re

log_path = f'{DRIVE_LOGS}/{DATASET}/runs/{EXP_NAME}/train.log'
lines = [l for l in open(log_path, errors='ignore')
         if 'val/macro' in l or 'train: loss_all' in l]
print(f'--- {log_path} ---')
for l in lines:
    print(re.sub(r'\x1b\[[0-9;]*m', '', l).rstrip())

vals = [l for l in lines if 'val/macro' in l]
if vals:
    print('\n=== 最終 val/macro ===')
    print(re.sub(r'\x1b\[[0-9;]*m', '', vals[-1]).strip())

---

## 12. 推論（3セット一括: val / 交差点シナリオ / レベル正規化プローブ）

予測CSVを1本にまとめて Drive に保存 → ローカルの解剖（step8系・step12通知層）が読む。

In [ ]:
import glob, os
best_ckpt = sorted(glob.glob(f'{DRIVE_LOGS}/{DATASET}/runs/{EXP_NAME}/checkpoints/epoch_*.ckpt'))[-1]
print('using:', best_ckpt)

for tag in ['valinfer', 'scenario', 'probe']:
    short = {'valinfer': 'val', 'scenario': 'scenario', 'probe': 'probe'}[tag]
    exp = f'infer_{EXP_NAME}_{short}'
    !python src/infer.py experiment=outdoor_siren_v9_{tag} \
        mode=test \
        ckpt_path="{best_ckpt}" \
        model.kwargs.pretrained_path=null \
        experiment_name={exp} \
        paths.log_dir={DRIVE_LOGS}
    sub = f'{DRIVE_LOGS}/{DATASET}/runs/{exp}/submissions'
    out_lines = []
    for p in sorted(glob.glob(f'{sub}/*.csv')):
        stem = os.path.basename(p)[:-4]
        for line in open(p):
            if line.strip():
                out_lines.append(f'{stem},{line.strip()}')
    out = f'{DRIVE_DATA}/{exp}_all.csv'
    open(out, 'w').write('\n'.join(out_lines))
    print('wrote', out, len(out_lines), 'lines')

## 13. このあと（ローカル側）

1. 3つの `infer_..._all.csv` をローカルへ → 解剖（クラス別・層別・可聴マスク上の見逃し）
2. step12（通知層）: ルールv1 でリードタイム採点＋オラクル上限の併記
3. プローブ枠で「音量込み vs 音量正規化」の2段報告（案A''の証明）
4. test(fold3) は全分析が固まった後に**最終1回だけ**